# Time-Travel with LangSmithAPI

## Steps: 
1. Have Agent.py file ready in the studio folder
2. keep .env file with LangSmith API keys and GOOGle API keys ready
3. set langgraph.json file ready
4. To start the local development server,
   run the following command in your terminal in the /studio directory in this module: BAsh
   ' langgraph dev '

### check for output:
 API: http://127.0.0.1:2024
- 🎨 Studio UI: https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024
- 📚 API Docs: http://127.0.0.1:2024/docs

This in-memory server is designed for development and testing.
For production use, please use LangSmith Deployment.

6. Copy url = http://127.0.0.1:2024

#### Note: 
- check for more info : https://docs.langchain.com/langsmith/quick-start-studio#local-development-server
- check for LangSmith setup guide in the begining of the course.

7. ### Connecting to the Local Server Using the SDK
You use the LangGraph SDK to talk to your local agent: python
```
from langgraph_sdk import get_client

url = "http://127.0.0.1:2024"
client = get_client(url=url)
```
This client object lets you:

    - list assistants
    - create threads
    - run your agent
    - stream results

8. ### Create a thread for storing event checkpoints?
A thread is a conversation session.

It stores:

    - messages
    - memory
    - state
    - checkpoints

You create one like this: python
```
thread = await client.threads.create()
```
This returns: python
```
{"thread_id": "abc123"}
```
You will use this thread ID for the entire conversation.

In [1]:
from langgraph_sdk import get_client

url = "http://127.0.0.1:2024"
client  = get_client(url=url)

# Search all hosted graphs
assistants = await client.assistants.search()
# list the agents
assistants

[{'assistant_id': 'fe096781-5601-53d2-b2f6-0d3403f7e9ca',
  'graph_id': 'agent',
  'config': {},
  'context': {},
  'metadata': {'created_by': 'system'},
  'name': 'agent',
  'created_at': '2026-05-04T19:40:44.199438+00:00',
  'updated_at': '2026-05-04T19:40:44.199438+00:00',
  'version': 1,
  'description': None},
 {'assistant_id': '09f72e2a-1f9e-53ce-9257-2a01f9362e09',
  'graph_id': 'Dynamic_Breakpoints',
  'config': {},
  'context': {},
  'metadata': {'created_by': 'system'},
  'name': 'Dynamic_Breakpoints',
  'created_at': '2026-05-01T21:39:15.638281+00:00',
  'updated_at': '2026-05-01T21:39:15.638281+00:00',
  'version': 1,
  'description': None}]

In [9]:
# create a thread
thread = await client.threads.create()
print(thread)

{'thread_id': '019df4c9-a762-7c31-be3b-f5c9e255a4d5', 'created_at': '2026-05-04T20:59:09.538609+00:00', 'updated_at': '2026-05-04T20:59:09.538609+00:00', 'state_updated_at': '2026-05-04T20:59:09.538609+00:00', 'metadata': {}, 'status': 'idle', 'config': {}, 'values': None}


In [10]:
# stream events
from langchain_core.messages import HumanMessage
msg_input = [ HumanMessage(content="Divide 30 and 10")]
async for event in client.runs.stream(
        thread["thread_id"],   # which conversation
        assistant_id= "agent",               # which assistant
        input={"messages": msg_input},       # what the user said
        stream_mode="updates",  # stream full state after each step
            ):
    print(event)
    #capture messages
    #messages = event.data.get('messages',[])
    # if messages exist
    #if messages:
        #print("\n", messages[-1]) # print last message in the list
    print("-" * 50)

StreamPart(event='metadata', data={'run_id': '019df4c9-ae2d-7ab1-bb44-60a9f5f491ba', 'attempt': 1}, id=None)
--------------------------------------------------
StreamPart(event='updates', data={'llm_assistant': {'messages': [{'content': [], 'additional_kwargs': {'function_call': {'name': 'division', 'arguments': '{"b": 10, "a": 30}'}, '__gemini_function_call_thought_signatures__': {'ed5ca7d5-283d-4610-bad6-6f7bac5068a3': 'EjQKMgEMOdbHxWC6jEA3Wa/elvx5mWrCSZ1FWwGagHPh/SZWtlS+qPnz+Lv7Q/PWJ78nZ8uV'}}, 'response_metadata': {'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, 'type': 'ai', 'name': None, 'id': 'lc_run--019df4c9-b169-7cc1-98cd-5d87e3639144-0', 'tool_calls': [{'name': 'division', 'args': {'b': 10, 'a': 30}, 'id': 'ed5ca7d5-283d-4610-bad6-6f7bac5068a3', 'type': 'tool_call'}], 'invalid_tool_calls': [], 'usage_metadata': {'input_tokens': 254, 'output_tokens': 18, 'total_tokens': 272, 'input_token_details'

## get current state

In [11]:
# current_state 
current_state = await client.threads.get_state(thread['thread_id'])
current_state

{'values': {'messages': [{'content': 'Divide 30 and 10',
    'additional_kwargs': {},
    'response_metadata': {},
    'type': 'human',
    'name': None,
    'id': 'fb90c865-8d07-4dd1-8f53-82a5fa02d4b9'},
   {'content': [],
    'additional_kwargs': {'function_call': {'name': 'division',
      'arguments': '{"b": 10, "a": 30}'},
     '__gemini_function_call_thought_signatures__': {'ed5ca7d5-283d-4610-bad6-6f7bac5068a3': 'EjQKMgEMOdbHxWC6jEA3Wa/elvx5mWrCSZ1FWwGagHPh/SZWtlS+qPnz+Lv7Q/PWJ78nZ8uV'}},
    'response_metadata': {'finish_reason': 'STOP',
     'model_name': 'gemini-3.1-flash-lite-preview',
     'safety_ratings': [],
     'model_provider': 'google_genai'},
    'type': 'ai',
    'name': None,
    'id': 'lc_run--019df4c9-b169-7cc1-98cd-5d87e3639144-0',
    'tool_calls': [{'name': 'division',
      'args': {'b': 10, 'a': 30},
      'id': 'ed5ca7d5-283d-4610-bad6-6f7bac5068a3',
      'type': 'tool_call'}],
    'invalid_tool_calls': [],
    'usage_metadata': {'input_tokens': 254,
    

## Replay:

- To replay , get the checkpoint id or config using **get state_history**
- First checkpoint listed is the current state
- get the assistant node checkpoint

In [12]:
state_history = await client.threads.get_history(thread['thread_id'])

# get checkpoint created at assistant node  - next = assistant
replay_state = state_history[-2]
replay_state

{'values': {'messages': [{'content': 'Divide 30 and 10',
    'additional_kwargs': {},
    'response_metadata': {},
    'type': 'human',
    'name': None,
    'id': 'fb90c865-8d07-4dd1-8f53-82a5fa02d4b9'}]},
 'next': ['llm_assistant'],
 'tasks': [{'id': '2f924984-69d8-c88f-7572-7a4c7c9291c5',
   'name': 'llm_assistant',
   'path': ['__pregel_pull', 'llm_assistant'],
   'error': None,
   'interrupts': [],
   'checkpoint': None,
   'state': None,
   'result': {'messages': [{'content': [],
      'additional_kwargs': {'function_call': {'name': 'division',
        'arguments': '{"b": 10, "a": 30}'},
       '__gemini_function_call_thought_signatures__': {'ed5ca7d5-283d-4610-bad6-6f7bac5068a3': 'EjQKMgEMOdbHxWC6jEA3Wa/elvx5mWrCSZ1FWwGagHPh/SZWtlS+qPnz+Lv7Q/PWJ78nZ8uV'}},
      'response_metadata': {'finish_reason': 'STOP',
       'model_name': 'gemini-3.1-flash-lite-preview',
       'safety_ratings': [],
       'model_provider': 'google_genai'},
      'type': 'ai',
      'name': None,
      'i

### Get checkpoint

In [13]:
checkpoint_id = replay_state['checkpoint_id']
checkpoint_id

'1f147fc1-a212-6c19-8000-2787f9a55d7f'

### replay from a specified checkpoint id

In [14]:
async for chunk in client.runs.stream(
    thread['thread_id'],
    assistant_id='agent',
    input = None,
    stream_mode="updates",
    checkpoint_id=checkpoint_id
):
    print(chunk.data)
    #messages = chunk.data.get('messages',[])
    #print(messages)
    print("\n")

{'run_id': '019df4ca-bf86-7ac2-b2f1-6925aa355172', 'attempt': 1}


{'llm_assistant': {'messages': [{'content': [], 'additional_kwargs': {'function_call': {'name': 'division', 'arguments': '{"a": 30, "b": 10}'}, '__gemini_function_call_thought_signatures__': {'5a94a510-ae3e-4e3b-a91c-0ae04d80a9b6': 'EjQKMgEMOdbHkQQHics3RptSgVj01h2tjdbW48mDxq0S+VTlrbUc66QmUS5A+QJMqBVKgbYk'}}, 'response_metadata': {'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, 'type': 'ai', 'name': None, 'id': 'lc_run--019df4ca-c354-7152-a243-d0e084e2323d-0', 'tool_calls': [{'name': 'division', 'args': {'a': 30, 'b': 10}, 'id': '5a94a510-ae3e-4e3b-a91c-0ae04d80a9b6', 'type': 'tool_call'}], 'invalid_tool_calls': [], 'usage_metadata': {'input_tokens': 254, 'output_tokens': 18, 'total_tokens': 272, 'input_token_details': {'cache_read': 0}}}]}}


{'tools': {'messages': [{'content': '3.0', 'additional_kwargs': {}, 'response_metadata': {}, 'type'

In [15]:
# current_state 
current_state = await client.threads.get_state(thread['thread_id'])
current_state

{'values': {'messages': [{'content': 'Divide 30 and 10',
    'additional_kwargs': {},
    'response_metadata': {},
    'type': 'human',
    'name': None,
    'id': 'fb90c865-8d07-4dd1-8f53-82a5fa02d4b9'},
   {'content': [],
    'additional_kwargs': {'function_call': {'name': 'division',
      'arguments': '{"a": 30, "b": 10}'},
     '__gemini_function_call_thought_signatures__': {'5a94a510-ae3e-4e3b-a91c-0ae04d80a9b6': 'EjQKMgEMOdbHkQQHics3RptSgVj01h2tjdbW48mDxq0S+VTlrbUc66QmUS5A+QJMqBVKgbYk'}},
    'response_metadata': {'finish_reason': 'STOP',
     'model_name': 'gemini-3.1-flash-lite-preview',
     'safety_ratings': [],
     'model_provider': 'google_genai'},
    'type': 'ai',
    'name': None,
    'id': 'lc_run--019df4ca-c354-7152-a243-d0e084e2323d-0',
    'tool_calls': [{'name': 'division',
      'args': {'a': 30, 'b': 10},
      'id': '5a94a510-ae3e-4e3b-a91c-0ae04d80a9b6',
      'type': 'tool_call'}],
    'invalid_tool_calls': [],
    'usage_metadata': {'input_tokens': 254,
    

## When you click Replay (or re-run a thread via API), LangGraph does not mutate the old run.
Instead, it:
    - Creates a new run ID 
    - Restores the old run’s initial checkpoint
    - Executes the graph again
    - Saves new checkpoints for this new run

##### So you end up with:
    - Old run → original checkpoints
    - New run → new checkpoints created during replay
    - This is intentional. It guarantees:

##### Reproducibility: - Isolation (your replay cannot corrupt the original run)
                 - Deterministic time‑travel (each run has its own timeline)

### 🔍 What actually happens under the hood
##### When you replay:
- LangGraph loads the saved state from the original checkpoint
- It starts a new execution context
- Every node that runs writes a new checkpoint entry into the checkpointer
- original run created checkpoints:
```
C0 → C1 → C2 → C3
```
#### When you replay from C2:
```
Original run:   C0 → C1 → C2 → C3
Replay run:     C2' → C3' → C4'
```
#### Notice:
- C2' is a copy of C2
- C3' and C4' are new checkpoints created during replay

In [ ]:
# Fork

# Fork
-  Forking: Creating a new run or thread starting from an existing one, usually with small changes (prompt, config, code).
- Create a new thread
- use messases.id to modify the content
  

In [19]:
initial_input = {"messages": HumanMessage(content="Multiply 2 and 3")}
thread = await client.threads.create()
async for chunk in client.runs.stream(
    thread["thread_id"],
    assistant_id="agent",
    input=initial_input,
    stream_mode="updates",
):
    print(chunk.data)
    print("\n")

{'run_id': '019df51d-9eb2-78d2-8463-bdc82ab4aa9c', 'attempt': 1}


{'llm_assistant': {'messages': [{'content': [], 'additional_kwargs': {'function_call': {'name': 'multiply', 'arguments': '{"a": 2, "b": 3}'}, '__gemini_function_call_thought_signatures__': {'d42c7726-9e28-4484-8ff2-9dd3b4864811': 'EjQKMgEMOdbHrEBHzsAOqsNmYZcypfqjAg1WyWJ/6Dgnd6yUGmphacIJIN1ZNJCmia4E/MFU'}}, 'response_metadata': {'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, 'type': 'ai', 'name': None, 'id': 'lc_run--019df51d-a127-7cc1-a8ea-e9ca8489726b-0', 'tool_calls': [{'name': 'multiply', 'args': {'a': 2, 'b': 3}, 'id': 'd42c7726-9e28-4484-8ff2-9dd3b4864811', 'type': 'tool_call'}], 'invalid_tool_calls': [], 'usage_metadata': {'input_tokens': 252, 'output_tokens': 16, 'total_tokens': 268, 'input_token_details': {'cache_read': 0}}}]}}


{'tools': {'messages': [{'content': '6.0', 'additional_kwargs': {}, 'response_metadata': {}, 'type': 't

### get state history

In [22]:
state_history =  await client.threads.get_history(thread['thread_id'])
to_fork = state_history[-2]
to_fork

{'values': {'messages': [{'content': 'Multiply 2 and 3',
    'additional_kwargs': {},
    'response_metadata': {},
    'type': 'human',
    'name': None,
    'id': '5451d309-d577-43e0-b369-8770c5810d3f'}]},
 'next': ['llm_assistant'],
 'tasks': [{'id': 'ddf551fe-5b21-7990-763a-85ff9487b4eb',
   'name': 'llm_assistant',
   'path': ['__pregel_pull', 'llm_assistant'],
   'error': None,
   'interrupts': [],
   'checkpoint': None,
   'state': None,
   'result': {'messages': [{'content': [],
      'additional_kwargs': {'function_call': {'name': 'multiply',
        'arguments': '{"a": 2, "b": 3}'},
       '__gemini_function_call_thought_signatures__': {'d42c7726-9e28-4484-8ff2-9dd3b4864811': 'EjQKMgEMOdbHrEBHzsAOqsNmYZcypfqjAg1WyWJ/6Dgnd6yUGmphacIJIN1ZNJCmia4E/MFU'}},
      'response_metadata': {'finish_reason': 'STOP',
       'model_name': 'gemini-3.1-flash-lite-preview',
       'safety_ratings': [],
       'model_provider': 'google_genai'},
      'type': 'ai',
      'name': None,
      'id'

In [23]:
# checkpoint id
checkpoint_id = to_fork['checkpoint_id']
checkpoint_id

'1f14808e-8e60-6fc3-8000-deed910594b3'

In [24]:
# message_id 
msg_id = to_fork['values']['messages'][0]['id']
msg_id

'5451d309-d577-43e0-b369-8770c5810d3f'

# edit / Modify state
Let's edit the state.
Remember how our reducer on `messages` works: 
* It will append, unless we supply a message ID.
* We supply the message ID to overwrite the message, rather than appending to state!

In [25]:
# add id to the input message to modify the original input
forked_input = {"messages": HumanMessage(content="divide 10 and 3",
                                         id=msg_id)}

forked_config = await client.threads.update_state(
    thread["thread_id"],
    forked_input,
    checkpoint_id=checkpoint_id
)
forked_config

{'checkpoint': {'thread_id': '019df51d-9eae-7d42-81e6-f50c48cea0a5',
  'checkpoint_ns': '',
  'checkpoint_id': '1f1480a3-04ee-6629-8001-fa5720c30232'},
 'configurable': {'thread_id': '019df51d-9eae-7d42-81e6-f50c48cea0a5',
  'checkpoint_ns': '',
  'checkpoint_id': '1f1480a3-04ee-6629-8001-fa5720c30232'},
 'checkpoint_id': '1f1480a3-04ee-6629-8001-fa5720c30232'}

In [26]:
cur_state = await client.threads.get_state(thread['thread_id'])
cur_state

{'values': {'messages': [{'content': 'divide 10 and 3',
    'additional_kwargs': {},
    'response_metadata': {},
    'type': 'human',
    'name': None,
    'id': '5451d309-d577-43e0-b369-8770c5810d3f'}]},
 'next': ['llm_assistant'],
 'tasks': [{'id': '39e4e264-c0e0-6515-0e91-b1d3c7e0d364',
   'name': 'llm_assistant',
   'path': ['__pregel_pull', 'llm_assistant'],
   'error': None,
   'interrupts': [],
   'checkpoint': None,
   'state': None,
   'result': None}],
 'metadata': {'graph_id': 'agent',
  'thread_id': '019df51d-9eae-7d42-81e6-f50c48cea0a5',
  'checkpoint_id': '1f14808e-8e60-6fc3-8000-deed910594b3',
  'source': 'update',
  'step': 1,
  'parents': {}},
 'created_at': '2026-05-04T22:40:02.254596+00:00',
 'checkpoint': {'checkpoint_id': '1f1480a3-04ee-6629-8001-fa5720c30232',
  'thread_id': '019df51d-9eae-7d42-81e6-f50c48cea0a5',
  'checkpoint_ns': ''},
 'parent_checkpoint': {'checkpoint_id': '1f14808e-8e60-6fc3-8000-deed910594b3',
  'thread_id': '019df51d-9eae-7d42-81e6-f50c48c

In [28]:
# next node shows that graph is yet to run
cur_state['next']

['llm_assistant']

# Re-run the graph with new checkpoint ID created while branching

In [29]:
async for chunk in client.runs.stream(
    thread["thread_id"],
    assistant_id="agent",
    input=None,
    stream_mode="updates",
    checkpoint_id= forked_config['checkpoint_id']
):
    print(chunk.data)
    print("\n")

{'run_id': '019df61d-ce3f-7141-a2ff-758125072a77', 'attempt': 1}


{'llm_assistant': {'messages': [{'content': [], 'additional_kwargs': {'function_call': {'name': 'division', 'arguments': '{"b": 3, "a": 10}'}, '__gemini_function_call_thought_signatures__': {'f7b07223-5b56-4738-808c-6f12b3d39c6d': 'EjQKMgEMOdbHnI8oaylXavdFsHD03tgFXOuXUDpHY3Je1q4iiPoDZGEWQB96uAO5HhWd8qwR'}}, 'response_metadata': {'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, 'type': 'ai', 'name': None, 'id': 'lc_run--019df61d-d1e5-70e0-a741-bad44b9be4e5-0', 'tool_calls': [{'name': 'division', 'args': {'b': 3, 'a': 10}, 'id': 'f7b07223-5b56-4738-808c-6f12b3d39c6d', 'type': 'tool_call'}], 'invalid_tool_calls': [], 'usage_metadata': {'input_tokens': 253, 'output_tokens': 17, 'total_tokens': 270, 'input_token_details': {'cache_read': 0}}}]}}


{'tools': {'messages': [{'content': '3.3333333333333335', 'additional_kwargs': {}, 'response_metadata